In [ ]:
import numpy as np
import pandas as pd
import csv
import scipy.stats as sci
import sklearn
#from sklearn.metrics import average_precision_score
#from sklearn.metrics import matthews_corrcoef
#from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
import matplotlib.pyplot as plt
from sklearn import metrics
import pickle


def load_data(SAE_type, model, layer, mask):

    # Load feature data and encodings
    dssp_path = '/home/neelm/SAEProteinMPNN/evaluation/created_data/features/node_features.csv'
    encodings_path = '/home/neelm/SAEProteinMPNN/evaluation/created_data/encodings/' + SAE_type + '_' + model + '/output_' + model + '_' + str(layer) + '.pkl'

    # Read and format pkl files
    dfs = []
    with open(encodings_path, "rb") as f:
        while True:
            try:
                dfs.append(pickle.load(f))
            except EOFError:
                break
        encodings = pd.concat(dfs, ignore_index=True)
    dfs = []
    np.random.seed(0)
    mask = np.random.rand(encodings.shape[0]) < mask_lvl
    encodings = encodings.iloc[mask, :]

    dssp = pd.read_csv(dssp_path)
    print("read data")
    # Filter data to make sure there are entries for both sets of data and remove duplicates
    encodings = encodings.drop_duplicates(subset='identifier')
    dssp = dssp.drop_duplicates(subset='identifier')
    encodings = encodings.dropna() # Drop rows with NA values
    dssp = dssp.dropna()
    encodings['identifier'] = encodings['identifier'].astype(str).str.strip().str.lower()
    dssp['identifier'] = dssp['identifier'].astype(str).str.strip().str.lower()
    matching_ids = set(encodings['identifier']) & set(dssp['identifier'])
    encodings = encodings[encodings['identifier'].isin(matching_ids)]
    encodings = encodings.sort_values('identifier').reset_index(drop=True)
    dssp = dssp[dssp['identifier'].isin(matching_ids)]
    dssp = dssp.sort_values('identifier').reset_index(drop=True)
    print("filtered data")

    # Isolate sample labels and set up mask (default 10% of samples -> ~43,000 samples)
    res_labels = encodings.iloc[:, 0]
    encodings = encodings.iloc[:, 1:]
    print(f'{res_labels.shape[0]} samples')
    
    #dssp = dssp.iloc[mask, :]
    #res_labels = res_labels.iloc[mask]
    print(f'{res_labels.shape[0]} samples')
    n_dims = encodings.shape[1]
    print(f'{n_dims} dimensions')

    return encodings, dssp, res_labels, n_dims

# ROC AUC Thresh = 0.7
# Pearson Thresh = 0.5

alphabet = list('ACDEFGHIKLMNPQRSTVWY')  # Standard amino acids
# Anova test, not used in favor of ROC AUC
def ANOVA(feature_act, encodings, aa, neuron):

    feauture_present = encodings[feature_act]
    feature_missing = encodings[feature_act==False]
    f_stat, p_val = sci.f_oneway(feauture_present, feature_missing)

    if p_val < 1e-4 and f_stat > 5:
        print(f"{f_stat},{neuron},{aa}")

# Mutual information, not used in favor of ROC AUC
def mi_class(dssp, encodings, feat, dim):
    y = dssp.to_numpy()
    x = encodings.to_numpy().reshape(-1, 1)

    #mask = np.isfinite(x) & np.isfinite(y)
    #x = x[mask]
    #y = y[mask]

    #if np.sum(x) == 0 or np.sum(y) == 0:
    #    return 0
    #if np.std(x) == 0 or np.std(y) == 0:
    #    return 0
    
    mi = mutual_info_classif(x, y)
    if mi > 0.2:
        print(mi, dim)
        return 1
    else:
        #print(mi, dim)
        return 0

# F1 scores, not used in favor of ROC AUC
def get_f1_scores(thresh, feature_act):

    true_pos = (feature_act == True).sum()
    false_pos = (feature_act == False).sum()
    false_neg = 0
    default_f1 = true_pos/((false_pos + false_neg)* 0.5 + true_pos)

    true_pos = ((thresh == True) & (feature_act == True)).sum()
    false_pos = ((thresh == True) & (feature_act == False)).sum()
    false_neg = ((thresh == False) & (feature_act == True)).sum()

    f1 = true_pos/((false_pos + false_neg)* 0.5 + true_pos)

    return f1 #(f1 - default_f1)/(1 - default_f1)

# Mutual information, not used in favor of Pearson
def mi_regress(dssp, encodings, feat, dim):
    y = dssp.to_numpy()
    x = encodings.to_numpy().reshape(-1, 1)

    #mask = np.isfinite(x) & np.isfinite(y)
    #x = x[mask]
    #y = y[mask]

    if np.sum(x) == 0 or np.sum(y) == 0:
        return 0
    #if np.std(x) == 0 or np.std(y) == 0:
        return 0
    
    mi = mutual_info_regression(x, y)
    if mi > 0.2:
        print(mi, feat, dim)
        return 1
    else:
        return 0    

# Pearson correlation test
def safe_pearsonr(encodings, features, feature_name, dim, thresh=0.5):
    activation = encodings.iloc[:, dim].to_numpy()
    features = features.to_numpy()
    mask = np.isfinite(features) & np.isfinite(activation)

    features = features[mask]
    activation = activation[mask]

    if np.std(features) == 0 or np.std(activation) == 0:
        return 0

    r, p = sci.pearsonr(features, activation)
    if r > thresh or r < -1*thresh:
        print(r, dim, feature_name)
        return 1
    else:
        return 0

def graph_roc_auc():
    fpr, tpr, _ = sklearn.metrics.roc_curve(feature_act, activations, dim, model, feat)
    roc_auc = sklearn.metrics.auc(fpr, tpr)
    # Plot ROC
    plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--') # Baseline
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC AUC Curve for Presence of {feat}: Dim {dim}')
    plt.legend(loc='upper left')
    plt.savefig(f'{model}_roc_auc_{dim}.png')
    plt.show()

def roc_auc(encodings, features, feature_name, dim, thresh=0.6, graph=False):
    activations = encodings.iloc[:, dim]

    # Ensure equal number of true postitives/negatives
    pos_idx = pd.Series(features[features == True].index.tolist())
    neg_idx = pd.Series(features[features == False].index.tolist())
    n = min(len(pos_idx), len(neg_idx))
    #print(len(pos_idx), len(neg_idx))
    if n > 40:
        balanced_pos = pos_idx.sample(n, random_state=0).sort_values() # Randomly downsample
        balanced_neg = neg_idx.sample(n, random_state=0).sort_values()
        balanced_idx = balanced_pos.tolist() + balanced_neg.to_list()

        features = features.loc[balanced_idx]
        activations = activations.loc[balanced_idx]
        score = round(sklearn.metrics.roc_auc_score(features, activations), 3)
        if score > thresh or score < (1-thresh):
            print(feature_name, dim, score)
            return 1
        else:
            #print(feature_name, dim, score)
            return 0
    else:
        return 0

# Add ['Function'] and ['Shape'] columns to feature_df
def categorize_AA(feature_df):
    # By function
    function_dict = {
        'phobic': list('GAVILM'),
        'philic': list('CPSTNQ'),
        'pos': list('KRH'),
        'neg': list('DE'),
        'aro': list('FYW')
    }

    # By polarity (Trinquier and Sanejouand 1998)
    pol_dict = {
        'phobic': list('WCMIFLV'),
        'mid': list('GRSTAP'),
        'philic': list('EDKNQHY')
    } 

    # By shape
    shape_dict = {
        'small': list('GA'),
        'sbranch': list('SCVT'),
        'branch1': list('DNIL'),
        'branch2': list('EQ'),
        'long': list('KM'),
        'lbranch': list('R'),
        'ring1': list('PHFY'),
        'ring2': list('W')
    }

    # By volume
    # Cateogrized by Van der Waals volume of side chain (Darby and Creighton 1993)
    vol_dict = {
        'vsmall': list('G'), # 48
        'small': list('A'), # 67
        'larger': list('S'), # 73
        'smedium': list('CPDTN'), # 86-96
        'medium': list('VEQH'), # 105-118
        'lmedium': list('ILM'), # 124
        'large': list('KFYR'), # 135-148
        'vlarge': list('W') # 163
    }
    # Mapping residues
    residue_to_function = {
        residue: category
        for category, residues in function_dict.items()
        for residue in residues
    }

    residue_to_polarity = {
        residue: category
        for category, residues in pol_dict.items()
        for residue in residues
    }

    # Mapping for residue to shape
    residue_to_shape = {
        residue: category
        for category, residues in shape_dict.items()
        for residue in residues
    }

    residue_to_vol = {
        residue: category
        for category, residues in vol_dict.items()
        for residue in residues
    }
    feature_df.insert(loc = 3, column = 'Function', value = feature_df['residue'].map(residue_to_function))
    feature_df.insert(loc = 4, column = 'Polarity', value = feature_df['residue'].map(residue_to_polarity))
    feature_df.insert(loc = 5, column = 'Shape', value = feature_df['residue'].map(residue_to_shape))
    feature_df.insert(loc = 6, column = 'Volume', value = feature_df['residue'].map(residue_to_vol))
    #feature_df['Function'] = feature_df['residue'].map(residue_to_function)
    #feature_df['Shape'] = feature_df['residue'].map(residue_to_shape)


def count_categorical(features, base_feature, n_dims, encodings, bar_labels, bar_heights, sum=True):
    one_hot = pd.get_dummies(features[base_feature])
    counter, prev_total = 0, 0
    for feat in one_hot.columns:
        for dim in [7, 8, 46, 48, 163, 182, 191, 221]:#]range(n_dims):
            counter += roc_auc(encodings, one_hot[feat], feat, dim)
        print(feat)
        print(counter - prev_total)
        if sum != True:
            bar_labels.append(feat)
            bar_heights.append(counter / n_dims)
            counter = 0
        else:
            prev_total = counter
    if sum:
        print(base_feature)
        print(counter)
        bar_labels.append(base_feature)
        bar_heights.append(counter / n_dims)

def count_sig_dims(n_dims, encodings, features):

    bar_labels = []
    bar_heights = []
    #count_categorical(features, 'sec_struct', n_dims, encodings, bar_labels, bar_heights, sum=False)
    count_categorical(features, 'residue', n_dims, encodings, bar_labels, bar_heights, sum=False)
    #count_categorical(features, 'Function', n_dims, encodings, bar_labels, bar_heights)
    count_categorical(features, 'Polarity', n_dims, encodings, bar_labels, bar_heights)
    #count_categorical(features, 'Shape', n_dims, encodings, bar_labels, bar_heights)
    #count_categorical(features, 'Volume', n_dims, encodings, bar_labels, bar_heights)

    '''
    for feat in features.columns[7:]:
        counter = 0
        for dim in range(n_dims):
            counter += safe_pearsonr(encodings, features[feat], feat, dim)
        print(feat)
        print(counter)
        bar_labels.append(feat)
        bar_heights.append(counter / n_dims)
    '''
    return bar_labels, bar_heights


data_dict = {}
SAE_type = 'node'
models = ["log17_exp2"]
layer = 2 # 0-indexed
mask_lvl = 0.01

for model in models:
    encodings, dssp, res_labels, n_dims = load_data(SAE_type, model, layer, mask_lvl)
    categorize_AA(dssp)
    bar_labels, bar_heights = count_sig_dims(n_dims, encodings, dssp)
    data_dict[model] = bar_heights

'''
features = dssp['3']
for dim in [11, 13, 22, 78, 84, 95]:
    activation = encodings.iloc[:, dim]
    fig, ax = plt.subplots()
    ax = plt.scatter(activation, features)
    plt.xlabel('Sparse Activation')
    plt.ylabel(f'{feat}')
    plt.title(f'{dim}')
    plt.annotate("r\u00B2 = {:.3f}".format(rscore[idx]), (0, 1))
    plt.savefig(f"{dim}_{feat}_{model}scatterplot.png", dpi=500)
    plt.show()
'''


read data
filtered data
4304 samples
4304 samples
256 dimensions
A 7 0.432
A 46 0.552
A 182 0.43
A 191 0.368
A
4
C 8 0.618
C 46 0.579
C 48 0.365
C 182 0.561
C 191 0.657
C 221 0.75
C
6
D 7 0.411
D 48 0.447
D 163 0.698
D 182 0.323
D 191 0.369
D 221 0.377
D
6
E 7 0.401
E 8 0.387
E 182 0.412
E 191 0.339
E 221 0.353
E
5
F 7 0.563
F 8 0.439
F 163 0.367
F 191 0.589
F 221 0.614
F
5
G 7 0.756
G 8 0.576
G 48 0.337
G 163 0.568
G 182 0.289
G 191 0.365
G 221 0.357
G
7
H 8 0.426
H 182 0.375
H 221 0.39
H
3
I 7 0.642
I 8 0.661
I 48 0.594
I 163 0.334
I 182 0.874
I 191 0.778
I 221 0.807
I
7
K 7 0.423
K 8 0.38
K 46 0.424
K 191 0.367
K 221 0.318
K
5
L 46 0.727
L 48 0.807
L 163 0.411
L 182 0.632
L 191 0.669
L 221 0.75
L
6
M 48 0.675
M 163 0.346
M 191 0.642
M 221 0.671
M
4
N 48 0.442
N 163 0.744
N 182 0.392
N 221 0.417
N
4
P 8 0.313
P 48 0.292
P 163 0.424
P 182 0.312
P 191 0.417
P 221 0.361
P
6
Q 7 0.315
Q 8 0.429
Q 46 0.429
Q 163 0.557
Q 182 0.431
Q 191 0.368
Q 221 0.353
Q
7
R 7 0.413
R 8 0.412
R 221 0.383

'\nfeatures = dssp[\'3\']\nfor dim in [11, 13, 22, 78, 84, 95]:\n    activation = encodings.iloc[:, dim]\n    fig, ax = plt.subplots()\n    ax = plt.scatter(activation, features)\n    plt.xlabel(\'Sparse Activation\')\n    plt.ylabel(f\'{feat}\')\n    plt.title(f\'{dim}\')\n    plt.annotate("r² = {:.3f}".format(rscore[idx]), (0, 1))\n    plt.savefig(f"{dim}_{feat}_{model}scatterplot.png", dpi=500)\n    plt.show()\n'

In [ ]:
def clean_bar_graph(labels, data):
    rename = {
        'H': 'Alpha helix',
        'E': 'Beta strand',
        'B': 'Beta bridge',
        'G': '3-10 helix',
        'I': 'Pi helix',
        'T': 'Turn',
        'S': 'Bend',
        '-': 'None',
        'Function': 'Function',
        'Polarity': 'Polarity',
        'Shape': 'Shape',
        'Volume': 'Volume',
        'ASA': 'Exposed area',
        'phi': 'Phi angle',
        'psi': 'Psi angle',
        '0': '2.0 \u212B',
        '1': '3.3 \u212B',
        '2': '4.7 \u212B',
        '3': '6.0 \u212B',
        '4': '7.3 \u212B',
        '5': '8.7 \u212B',
        '6': '10.0 \u212B',
    }

    df = pd.DataFrame({}, index=rename.values())
    for model in data.keys():
        new_labels = []
        new_heights = []
        heights = data[model]
        height_dict = dict(zip(labels, heights))
        for label, new_label in rename.items():
            if label in height_dict:
                #new_labels.append(new_label)
                new_heights.append(height_dict[label] * 100)
        df[model] = new_heights
        print(model, new_heights)

    '''
    # Layer 1:
    dense = [0.0, 14.84375, 0.0, 0.0, 0.0, 1.5625, 0.0, 0.0, 0.0, 10.15625, 1.5625, 3.90625, 42.1875, 0.0, 0.0, 0.0, 0.0, 3.90625, 28.90625, 0.0, 21.09375, 5.46875]
    log20 = [64.84375, 13.28125, 5.46875, 10.15625, 0.0, 15.625, 25.0, 17.96875, 0.0, 0.0, 0.78125, 50.0, 0.0, 7.8125, 7.8125, 0.0, 0.78125, 0.0, 0.0, 0.0, 0.0, 0.0]
    log17 = [55.859375, 15.234375, 5.46875, 8.203125, 0.0, 19.140625, 21.09375, 16.015625, 0.0, 0.0, 0.390625, 55.859375, 0.0, 3.90625, 2.34375, 0.0, 1.953125, 0.0, 0.0, 0.0, 0.0, 0.0]
    log18 = [43.75, 9.765625, 2.9296875, 5.2734375, 0.0, 19.3359375, 20.1171875, 14.6484375, 0.0, 0.0, 0.0, 49.21875, 0.1953125, 2.34375, 0.9765625, 0.0, 0.78125, 0.1953125, 0.0, 0.1953125, 0.0, 0.0]
    log19 = [39.74609375, 5.95703125, 8.69140625, 8.0078125, 0.0, 17.7734375, 21.38671875, 13.37890625, 0.0, 0.09765625, 0.9765625, 45.8984375, 0.0, 0.87890625, 0.68359375, 0.0, 1.171875, 0.09765625, 0.0, 0.0, 0.0, 0.0]
    '''
    
    '''
    #Layer 2:
    dense = [0.0, 14.0625, 0.0, 0.0, 0.0, 1.5625, 0.0, 0.0, 0.0, 10.9375, 1.5625, 3.90625, 45.3125, 0.0, 0.0, 0.0, 0.0, 3.90625, 31.25, 0.0, 27.34375, 4.6875]
    log20 = [48.4375, 8.59375, 12.5, 6.25, 0.0, 6.25, 9.375, 5.46875, 0.0, 1.5625, 3.125, 37.5, 0.78125, 3.125, 0.78125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    log17 = [24.609375, 4.296875, 7.03125, 3.90625, 0.0, 5.859375, 4.6875, 5.46875, 0.0, 1.171875, 1.5625, 16.796875, 0.78125, 0.78125, 0.390625, 0.0, 0.78125, 0.390625, 0.0, 0.0, 0.0, 0.0]
    log18 = [28.125, 3.90625, 5.2734375, 5.46875, 0.0, 4.8828125, 3.3203125, 6.0546875, 0.0, 0.1953125, 2.34375, 18.9453125, 0.0, 0.5859375, 0.78125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    log19 = [26.26953125, 1.46484375, 7.91015625, 2.83203125, 0.0, 3.125, 2.44140625, 7.12890625, 0.0, 0.0, 0.78125, 16.40625, 0.09765625, 0.1953125, 0.09765625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    '''
    
    '''
    #Layer 3:
    

    '''
    #df = pd.DataFrame({"Dense": dense, "SAE Size=128": log20, "SAE Size=256": log17, "SAE Size=512": log18, "SAE Size=1024": log19}, index=rename.values())
    
    blues = plt.cm.Blues(np.linspace(0.9, 0.3, 8))
    greens = plt.cm.Greens(np.linspace(0.9, 0.3, 4))
    reds = plt.cm.Reds(np.linspace(0.9, 0.3, 3))
    purples = plt.cm.Purples(np.linspace(0.9, 0.3, 7))


    bar_colors = np.concatenate((blues, greens, reds, purples))
    #plt.rcParams['axes.autolimit_mode'] = 'round_numbers'
    
    #fig, ax = plt.subplots(figsize=(10, 6))

    index = np.arange(len(new_labels))
    bar_width = 0.4

    #ax = df.plot.barh()
    ax = df.plot.barh(width=0.9, figsize=(10, 6))#, labels=["SAE Size=128", "SAE Size=256", "SAE Size=512", "SAE Size=1024"])
    #for i, model in enumerate(df.columns):
    #    ax.barh(index + offset, df[model], bar_width, label=model, tick_label=new_labels, align='center')
    
    #ax.xaxis.set_tick_params(pad=5)
    #ax.yaxis.set_tick_params(pad=10)
    #ax.set_yticklabels(rename.)
    #ax.set_yticks(index + bar_width / 2)
    ax.set_xlabel('Percent of Dimensions Correlated')
    ax.invert_yaxis()
    ax.set_title(f"Correlated Dimensions for Layer {layer+1} of SAE models")
    plt.savefig(f"all models layer {layer+1} bar plot of feature correlations.png", bbox_inches='tight', dpi=300)
    plt.show()
    plt.clf()


clean_bar_graph(bar_labels, data_dict)